# Phase 2: Model Quantization and CoT Fine-Tuning (QLoRA)
This notebook implements the hardware-constrained training pipeline. We load the base SLM using 4-bit NormalFloat (NF4) quantization to fit inside edge-hardware VRAM. We then apply Low-Rank Adaptation (LoRA) and fine-tune the model using the Supervised Fine-Tuning Trainer (SFTTrainer) on our CoT financial dataset.

In [1]:
!pip install transformers peft bitsandbytes accelerate trl datasets

In [ ]:
# 1. Update/Install necessary libraries

import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# --- 1. Load Processed Data ---
print("Loading dataset...")
dataset = load_from_disk("processed_data/finqa_cot")
# Select a subset if you want a quick test run; otherwise, use the full dataset
train_data = dataset.select(range(min(5000, len(dataset))))

# --- 2. Hardware-Optimized 4-bit Quantization ---
model_id = "microsoft/Phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # Perfect for RTX 3090/4090
)

# --- 3. Tokenizer Setup ---
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" 

# --- 4. Model Loading (Fixing the KeyError & Remote Code issues) ---
print("Loading Model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto",
    # CRITICAL: Do NOT use trust_remote_code=True for Phi-3 anymore.
    # The native implementation is more stable.
    trust_remote_code=False, 
    attn_implementation="eager" # Prevents Flash-Attention missing errors
)
model = prepare_model_for_kbit_training(model)

# --- 5. LoRA Adapter Configuration ---
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
peft_model = get_peft_model(model, lora_config)

# --- 6. Modern Training Configuration (TRL 0.12+) ---
# SFTConfig replaces standard TrainingArguments for dataset-specific settings
sft_config = SFTConfig(
    output_dir="./fingeo_slm_outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    bf16=True, # Use bf16 for 3090/4090 (better than fp16)
    report_to="none",
    save_strategy="no",
    
    # Modern parameter names:
    dataset_text_field="formatted_prompt",
    max_length=2048, 
    packing=False
)

# --- 7. Initialize Trainer and Train ---
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_data,
    args=sft_config,
    processing_class=tokenizer # Replacement for 'tokenizer' arg in newer TRL
)

print("Starting CoT Fine-Tuning...")
trainer.train()

# --- 8. Save the trained LoRA adapter ---
trainer.model.save_pretrained("fingeo-slm-adapter")
print("Training complete. LoRA adapter saved.")

/Users/uzmanarfan/.pyenv/versions/3.12.9/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Quantized Model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]